In [1]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt

import re
from sklearn.model_selection  import train_test_split  
from sklearn.feature_extraction.text import TfidfVectorizer 

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense



In [2]:
df=pd.read_csv('IMDB Dataset.csv')

In [3]:
print(df.isnull().sum())

review       0
sentiment    0
dtype: int64


In [4]:
df.columns=['review','sentiment']

In [5]:
def clean_text(text):
    text=str(text)
    text=text.lower()
    text=re.sub(r'<.*?>','',text)
    text=re.sub(r'a-zA-Z','',text)

    return text


df['review']=df['review'].apply(clean_text)

In [9]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()

df['sentiment'] = encoder.fit_transform(df['sentiment'])
print(df['sentiment'].value_counts())

sentiment
1    25000
0    25000
Name: count, dtype: int64


In [10]:
vectorizer=TfidfVectorizer()
X=vectorizer.fit_transform(df['review'])
y=df['sentiment']

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [12]:
model = Sequential()

model.add(Dense(128, activation='relu', input_shape=(X_train.shape[1],)))
model.add(Dense(64, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

c:\Users\HP\anaconda4\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │    13,322,752 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13,331,073 (50.85 MB)

 Trainable params: 13,331,073 (50.85 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_test, y_test)

Epoch 1/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 154s 121ms/step - accuracy: 0.8829 - loss: 0.2879 - val_accuracy: 0.9044 - val_loss: 0.2288
Epoch 2/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 216s 133ms/step - accuracy: 0.9688 - loss: 0.0905 - val_accuracy: 0.8937 - val_loss: 0.2964
Epoch 3/10
1161/1250 ━━━━━━━━━━━━━━━━━━━━ 36s 415ms/step - accuracy: 0.9918 - loss: 0.0249

In [ ]:
loss,acc=model.evaluate(X_test,y_test)
print("accuracy: ",acc)

In [ ]:
y_pred=model.predict(X_test)
y_pred = (y_pred > 0.5).astype(int)
y_pred = y_pred.flatten()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report



cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['negative', 'positive'],
    yticklabels=['negative', 'positive']
)

plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')

plt.show()

# Classification Report
print(classification_report(y_test, y_pred))





In [ ]:
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')

plt.title("Accuracy Graph")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.show()